
Semantic search using TF-IDF + TruncatedSVD + Normalizer in a Pipeline.

In [ ]:
NOTES = [
    ("Train/test split", "We always hold back part of the data as a test set."),
    ("Overfitting", "Overfitting is when a model memorises the training rows instead of learning the pattern."),
    ("Cross-validation", "Cross-validation splits the data into k folds."),
    ("Data leakage", "Leakage happens when information from the test set sneaks into training."),
    ("Decision tree", "A decision tree asks a series of yes or no questions."),
    ("Gini impurity", "Gini impurity measures how mixed a group is."),
    ("Random forest", "A random forest grows many decision trees."),
    ("Feature importance", "Feature importance ranks how much each column contributed."),
    ("k-means clustering", "k-means is unsupervised, meaning it has no labels."),
    ("Choosing k", "Use elbow method and silhouette score."),
    ("Feature engineering", "Feature engineering means creating better input columns."),
    ("Scaling", "Standard scaling rewrites each value as a z-score."),
    ("One-hot encoding", "One-hot encoding turns a text column into indicator columns."),
    ("The neuron", "A neuron computes a weighted sum plus bias and activation."),
    ("Activation functions", "Sigmoid and ReLU are common activations."),
    ("Neural network layers", "Networks contain input, hidden and output layers."),
    ("Training a network", "Forward pass, loss, backpropagation, gradient descent."),
    ("Learning rate", "Learning rate controls step size in gradient descent."),
    ("Early stopping", "Stop when validation performance stops improving."),
    ("Bag of words", "Counts occurrences of vocabulary words."),
    ("TF-IDF", "Weights words by frequency and rarity."),
    ("Embeddings", "Embeddings place similar meanings close together."),
    ("Cosine similarity", "Compares vector directions."),
    ("Semantic search", "Embeds documents and query then ranks by similarity.")
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
titles = [t for t,_ in NOTES]
texts = [txt for _,txt in NOTES]

svd_dim = min(100, len(texts)-1)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("svd", TruncatedSVD(n_components=svd_dim, random_state=42)),
    ("normalizer", Normalizer(copy=False))
])

doc_vectors = pipeline.fit_transform(texts)
print(doc_vectors.shape)

In [ ]:
def search_notes(query, top_k=5):
    query_vec = pipeline.transform([query])
    scores = cosine_similarity(query_vec, doc_vectors)[0]
    idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in idx:
        results.append({
            "title": titles[i],
            "score": float(scores[i]),
            "text": texts[i]
        })
    return results

search_notes("How do I prevent overfitting in a neural network?")

In [ ]:
query = "What is cosine similarity used for?"
for r in search_notes(query, top_k=3):
    print(f"\nTitle: {r['title']}")
    print(f"Score: {r['score']:.3f}")
    print(r['text'])